# Movie Rating Prediction (Final) — Clean, Organized Notebook

**Dataset:** `mymoviedb.csv` (TMDB-style metadata)

**Goal:** Predict whether a movie is **highly rated**:
- `High_Rated = 1` if `Vote_Average >= 7.0`
- else `0`

**Structure:**
1. Data loading & cleaning
2. **All data visualizations** (one block)
3. Train/test split & models
4. **Learning theory** (VC dimension, PAC bounds)
5. Results & report

> Run top-to-bottom once to generate a complete results table.

In [ ]:
# If you run into missing packages, install first:
# !pip install -r requirements.txt

import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

warnings.filterwarnings("ignore")

# Plot style
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.05)
PALETTE = sns.color_palette("Set2", 8)
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

DATA_DIR = Path(".")
CSV_PATH = DATA_DIR / "mymoviedb.csv"
RANDOM_STATE = 42
THRESHOLD = 7.0

from IPython.display import display

np.random.seed(RANDOM_STATE)

## 1) Data loading

In [ ]:
df_raw = pd.read_csv(CSV_PATH, engine="python", on_bad_lines="skip")
print(df_raw.shape)
df_raw.head(3)

## 2) Cleaning + target definition

We:
- coerce numeric columns
- parse `Release_Date`
- create `Release_Year`, log transforms
- create the binary label `High_Rated` using the threshold 7.0

In [ ]:
df = df_raw.copy()

# Parse / coerce
for col in ["Popularity", "Vote_Count", "Vote_Average"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["Release_Date"] = pd.to_datetime(df["Release_Date"], errors="coerce")
df["Release_Year"] = df["Release_Date"].dt.year

# Basic feature engineering
df["log_popularity"] = np.log1p(df["Popularity"].clip(lower=0))
df["log_vote_count"] = np.log1p(df["Vote_Count"].clip(lower=0))

# Target
df["High_Rated"] = (df["Vote_Average"] >= THRESHOLD).astype(int)

# Minimal null handling for modeling rows
needed = ["Vote_Average", "High_Rated", "Release_Year", "log_popularity", "log_vote_count", "Genre", "Original_Language", "Overview"]
df_model = df.dropna(subset=[c for c in needed if c in df.columns]).copy()

print("Rows raw:", len(df_raw))
print("Rows after dropna for modeling:", len(df_model))
print(df_model[["Vote_Average", "High_Rated"]].describe())

## 3) Data visualization

All exploratory plots in one place — run this cell before modeling.

In [ ]:
# --- Class balance summary ---
pos = int((df_model["High_Rated"] == 1).sum())
neg = int((df_model["High_Rated"] == 0).sum())
print(f">= {THRESHOLD}: {pos:,} ({pos/(pos+neg):.2%})  |  < {THRESHOLD}: {neg:,} ({neg/(pos+neg):.2%})  |  ratio {neg/pos:.2f}:1")

# --- All EDA plots (single figure) ---
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1) Rating distribution
axes[0, 0].hist(df_model["Vote_Average"], bins=35, color="#6366F1", edgecolor="white", alpha=0.9)
axes[0, 0].axvline(THRESHOLD, color="#EF4444", ls="--", lw=2, label=f"threshold = {THRESHOLD}")
axes[0, 0].set_title("Vote Average Distribution", fontweight="bold")
axes[0, 0].set_xlabel("Vote Average")
axes[0, 0].legend(fontsize=8)

# 2) Class balance donut
counts = df_model["High_Rated"].value_counts().sort_index()
axes[0, 1].pie(
    counts.values,
    labels=[f"Low (<{THRESHOLD})", f"High (≥{THRESHOLD})"],
    autopct="%1.1f%%",
    startangle=90,
    colors=["#94A3B8", "#22C55E"],
    wedgeprops={"width": 0.45, "edgecolor": "white"},
)
axes[0, 1].set_title("Class Balance", fontweight="bold")

# 3) Popularity vs rating
sample = df_model.sample(min(3000, len(df_model)), random_state=RANDOM_STATE)
sc = axes[0, 2].scatter(
    sample["log_popularity"], sample["Vote_Average"],
    c=sample["High_Rated"], cmap="RdYlGn", alpha=0.35, s=14,
)
axes[0, 2].axhline(THRESHOLD, color="#EF4444", ls="--", lw=1.5)
axes[0, 2].set_title("Popularity vs Rating", fontweight="bold")
axes[0, 2].set_xlabel("log(1 + Popularity)")
plt.colorbar(sc, ax=axes[0, 2], label="High_Rated")

# 4) Vote count vs rating
axes[1, 0].scatter(
    sample["log_vote_count"], sample["Vote_Average"],
    c=sample["High_Rated"], cmap="RdYlGn", alpha=0.35, s=14,
)
axes[1, 0].axhline(THRESHOLD, color="#EF4444", ls="--", lw=1.5)
axes[1, 0].set_title("Vote Count vs Rating", fontweight="bold")
axes[1, 0].set_xlabel("log(1 + Vote Count)")

# 5) Mean rating by release year
year_means = df_model.groupby("Release_Year")["Vote_Average"].mean().dropna()
axes[1, 1].plot(year_means.index, year_means.values, color="#2E8B57", lw=2)
axes[1, 1].axhline(THRESHOLD, color="#EF4444", ls="--", lw=1.5)
axes[1, 1].set_title("Mean Rating by Year", fontweight="bold")
axes[1, 1].set_xlabel("Release Year")

# 6) Top genres by mean rating
genre_exploded = (
    df_model[["Vote_Average", "Genre"]]
    .dropna(subset=["Genre"])
    .assign(_g=lambda d: d["Genre"].str.split(", "))
    .explode("_g")
)
genre_exploded["_g"] = genre_exploded["_g"].str.strip()
genre_rating = (
    genre_exploded.groupby("_g")["Vote_Average"]
    .agg(["mean", "count"])
    .query("count >= 40")
    .sort_values("mean", ascending=False)
    .head(10)
)
genre_rating["mean"].plot(kind="barh", ax=axes[1, 2], color="#F59E0B")
axes[1, 2].invert_yaxis()
axes[1, 2].set_title("Top Genres by Mean Rating (n≥40)", fontweight="bold")
axes[1, 2].set_xlabel("Mean Vote Average")

plt.suptitle("Exploratory Data Analysis — Movie Dataset", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / "01_eda_overview.png", dpi=160, bbox_inches="tight")
plt.show()

## 4) Train/test split

In [ ]:
def make_balanced_train(X_train: pd.DataFrame, y_train: pd.Series, random_state: int = 42):
    """Random undersampling of the majority class."""
    train = X_train.copy()
    train["y"] = y_train.values
    n_pos = int((train["y"] == 1).sum())
    n_neg = int((train["y"] == 0).sum())
    if n_pos == 0 or n_neg == 0:
        return X_train, y_train

    maj = 0 if n_neg > n_pos else 1
    minc = 1 - maj

    train_maj = train[train["y"] == maj]
    train_min = train[train["y"] == minc]

    train_maj_ds = train_maj.sample(n=len(train_min), random_state=random_state)
    train_bal = pd.concat([train_maj_ds, train_min], axis=0).sample(frac=1, random_state=random_state)

    yb = train_bal.pop("y")
    return train_bal, yb


# --- Metadata-only features (simple, fast, strong baseline) ---
# We'll one-hot a limited set of Genres and Languages to avoid a huge sparse matrix.
TOP_GENRES = 12
TOP_LANGS = 8

def add_simple_metadata_features(df_in: pd.DataFrame) -> pd.DataFrame:
    out = df_in.copy()

    # Multi-genre split
    genre_lists = out["Genre"].fillna("Unknown").astype(str).str.split(", ")
    top_genres = (
        genre_lists.explode().str.strip().value_counts().head(TOP_GENRES).index.tolist()
    )
    for g in top_genres:
        out[f"genre_{g}"] = genre_lists.apply(lambda xs, gg=g: int(gg in [z.strip() for z in xs]))

    top_langs = out["Original_Language"].fillna("Unknown").astype(str).value_counts().head(TOP_LANGS).index.tolist()
    for lang in top_langs:
        out[f"lang_{lang}"] = (out["Original_Language"].fillna("Unknown").astype(str) == lang).astype(int)

    return out


df_feat = add_simple_metadata_features(df_model)
feature_cols = [c for c in df_feat.columns if c in ["log_popularity", "log_vote_count", "Release_Year"] or c.startswith(("genre_", "lang_"))]

X = df_feat[feature_cols].copy()
y = df_feat["High_Rated"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

X_train_bal, y_train_bal = make_balanced_train(X_train, y_train, random_state=RANDOM_STATE)

print("Train class ratio (original):", y_train.value_counts(normalize=True).to_dict())
print("Train class ratio (balanced):", y_train_bal.value_counts(normalize=True).to_dict())
print("Test  class ratio:", y_test.value_counts(normalize=True).to_dict())

## 5) Evaluation helpers

In [ ]:
def eval_binary(y_true, y_pred, y_score=None):
    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred)),
    }
    if y_score is not None:
        out["roc_auc"] = float(roc_auc_score(y_true, y_score))
        out["pr_auc"] = float(average_precision_score(y_true, y_score))
    else:
        out["roc_auc"] = np.nan
        out["pr_auc"] = np.nan
    return out


def model_category(name: str) -> str:
    if "Baseline" in name:
        return "Baseline"
    if "metadata" in name.lower() and "minilm" not in name.lower():
        return "Metadata"
    if any(k in name.lower() for k in ["tfidf", "lstm", "minilm", "text"]):
        return "Language"
    return "Other"


def show_confusion(y_true, y_pred, title="Confusion matrix", save_name=None):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(4.8, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=["Pred: Low", "Pred: High"],
        yticklabels=["True: Low", "True: High"],
        ax=ax,
    )
    ax.set_title(title, fontweight="bold")
    plt.tight_layout()
    if save_name:
        fig.savefig(FIG_DIR / save_name, dpi=160, bbox_inches="tight")
    plt.show()


results = []
fitted_models = {}  # name -> (y_true, y_pred, y_score) for ROC/PR plots

## 6) Baselines

In [ ]:
# Baseline 1: Uniform random (50/50)
rng = np.random.default_rng(RANDOM_STATE)
y_pred_rand = rng.integers(0, 2, size=len(y_test))
results.append({"model": "Baseline: uniform random", **eval_binary(y_test, y_pred_rand)})

# Baseline 2: Always majority class (on training set)
maj = int(y_train.value_counts().idxmax())
y_pred_maj = np.full(len(y_test), maj)
results.append({"model": f"Baseline: majority ({maj})", **eval_binary(y_test, y_pred_maj)})

pd.DataFrame(results).sort_values(["balanced_accuracy", "f1"], ascending=False)

## 7) Models — metadata only

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Logistic Regression
log_reg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
])
log_reg.fit(X_train_bal, y_train_bal)

y_pred = log_reg.predict(X_test)
y_score = log_reg.predict_proba(X_test)[:, 1]
results.append({"model": "LogReg (metadata)", **eval_binary(y_test, y_pred, y_score)})
fitted_models["LogReg (metadata)"] = (y_test, y_pred, y_score)
show_confusion(y_test, y_pred, title="LogReg (metadata)", save_name="cm_logreg_metadata.png")

pd.DataFrame(results).sort_values("pr_auc", ascending=False)

In [ ]:
cb = None
try:
    from catboost import CatBoostClassifier

    cb = CatBoostClassifier(
        iterations=400,
        depth=6,
        learning_rate=0.05,
        verbose=0,
        random_state=RANDOM_STATE,
        auto_class_weights="Balanced",
    )
    cb.fit(X_train_bal, y_train_bal)

    y_pred = cb.predict(X_test).astype(int)
    y_score = cb.predict_proba(X_test)[:, 1]
    results.append({"model": "CatBoost (metadata)", **eval_binary(y_test, y_pred, y_score)})
    fitted_models["CatBoost (metadata)"] = (y_test, y_pred, y_score)
except Exception as e:
    print("CatBoost not available (skipping).", repr(e))

pd.DataFrame(results).sort_values("pr_auc", ascending=False)

## 8) Text models — TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier

# Rich text for language models (Title + Genre + Overview)
def build_rich_text(df_in: pd.DataFrame) -> pd.Series:
    return (
        "Title: " + df_in["Title"].fillna("").astype(str)
        + " | Genre: " + df_in["Genre"].fillna("").astype(str)
        + " | Overview: " + df_in["Overview"].fillna("").astype(str)
    )

rich_text = build_rich_text(df_feat)

# Same train/test indices as metadata models (strict comparison)
Xtr_t = rich_text.loc[X_train.index]
Xte_t = rich_text.loc[X_test.index]
ytr_t = y_train
yte_t = y_test

tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    stop_words="english",
)

# TF‑IDF + Logistic Regression
text_logreg = Pipeline([
    ("tfidf", tfidf),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
])
text_logreg.fit(Xtr_t, ytr_t)

pred = text_logreg.predict(Xte_t)
score = text_logreg.predict_proba(Xte_t)[:, 1]
results.append({"model": "TFIDF+LogReg (text)", **eval_binary(yte_t, pred, score)})
fitted_models["TFIDF+LogReg (text)"] = (yte_t, pred, score)

# TF‑IDF + Random Forest
text_rf = Pipeline([
    ("tfidf", tfidf),
    ("model", RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_split=2,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=1,
    )),
])
text_rf.fit(Xtr_t, ytr_t)

pred = text_rf.predict(Xte_t)
score = text_rf.predict_proba(Xte_t)[:, 1]
results.append({"model": "TFIDF+RF (text)", **eval_binary(yte_t, pred, score)})
fitted_models["TFIDF+RF (text)"] = (yte_t, pred, score)

pd.DataFrame(results).sort_values("pr_auc", ascending=False)

## 9) Language understanding models

### 9A) LSTM (ready to run)

If TensorFlow is missing: `pip install tensorflow`

In [ ]:
# Install (one-time) if needed:
# !pip install tensorflow

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models

In [ ]:
# Tokenize + pad
MAX_WORDS = 20000
MAX_LEN = 200

Xtr = Xtr_t.astype(str).tolist()
Xte = Xte_t.astype(str).tolist()

tok = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tok.fit_on_texts(Xtr)

seq_tr = tok.texts_to_sequences(Xtr)
seq_te = tok.texts_to_sequences(Xte)

Xtr_pad = pad_sequences(seq_tr, maxlen=MAX_LEN, padding="post", truncating="post")
Xte_pad = pad_sequences(seq_te, maxlen=MAX_LEN, padding="post", truncating="post")

ytr = np.asarray(ytr_t, dtype=np.int32)
yte = np.asarray(yte_t, dtype=np.int32)

model = models.Sequential([
    layers.Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN),
    layers.Bidirectional(layers.LSTM(64, dropout=0.2, recurrent_dropout=0.0)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

neg = int((ytr == 0).sum())
pos = int((ytr == 1).sum())
class_weight = {0: 1.0, 1: (neg / max(pos, 1))}

history = model.fit(
    Xtr_pad,
    ytr,
    validation_split=0.2,
    epochs=3,
    batch_size=64,
    class_weight=class_weight,
    verbose=1,
)

proba = model.predict(Xte_pad, batch_size=128).ravel()
pred = (proba >= 0.5).astype(int)

results.append({"model": "LSTM (text)", **eval_binary(yte, pred, proba)})
fitted_models["LSTM (text)"] = (yte, pred, proba)
show_confusion(yte, pred, title="LSTM (text)", save_name="cm_lstm_text.png")

pd.DataFrame(results).sort_values("pr_auc", ascending=False)

### 9B) MiniLM — pretrained language understanding (text only)

Encodes rich text (`Title | Genre | Overview`) with `all-MiniLM-L6-v2`, then trains Logistic Regression on semantic embeddings alone.

In [ ]:
# Install (one-time) if needed:
# !pip install sentence-transformers

from sentence_transformers import SentenceTransformer

ENCODER_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
encoder = SentenceTransformer(ENCODER_MODEL)

print("Encoding train text (full set)...")
emb_train = encoder.encode(Xtr_t.tolist(), batch_size=64, show_progress_bar=True)
print("Encoding test text...")
emb_test = encoder.encode(Xte_t.tolist(), batch_size=64, show_progress_bar=True)

minilm_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        C=2.0,
        random_state=RANDOM_STATE,
    )),
])
minilm_clf.fit(emb_train, ytr_t)

y_pred = minilm_clf.predict(emb_test)
y_score = minilm_clf.predict_proba(emb_test)[:, 1]
results.append({"model": "MiniLM+LogReg (language)", **eval_binary(yte_t, y_pred, y_score)})
fitted_models["MiniLM+LogReg (language)"] = (yte_t, y_pred, y_score)
show_confusion(yte_t, y_pred, title="MiniLM+LogReg (language)", save_name="cm_minilm_language.png")

pd.DataFrame(results).sort_values("pr_auc", ascending=False)

## 10) Learning theory (course topics)

Connects our experiment to **Theoretical Data Science** material:
- **VC dimension** — capacity of linear classifiers (LogReg / linear SVM)
- **PAC generalization bound** — worst-case true error from empirical error + VC
- **Generalization gap** — train vs test error (overfitting indicator)
- **Regularization** — L2 penalty in LogReg controls effective capacity

In [ ]:
# --- VC dimension helpers ---
def linear_vc_dimension(n_features: int) -> int:
    """Affine hyperplanes in R^d have VC dimension d + 1."""
    return n_features + 1

def vc_generalization_bound(empirical_error: float, vc_dim: int, n: int, delta: float = 0.05) -> float:
    """PAC/VC bound: R(h) <= R_hat(h) + sqrt(8/n * (VC*log(2en/VC) + log(4/delta)))"""
    if vc_dim <= 0 or n <= 0:
        return 1.0
    vc_term = vc_dim * math.log2(2 * math.e * n / vc_dim)
    penalty = math.sqrt(8.0 / n * (vc_term + math.log(4.0 / delta)))
    return min(1.0, empirical_error + penalty)

n_features = len(feature_cols)
n_train = len(y_train)
vc_linear = linear_vc_dimension(n_features)

vc_table = pd.DataFrame([
    {"Hypothesis class": "Logistic Regression (L2)", "VC / bound": vc_linear, "Type": "exact VC = d+1",
     "Note": f"Hyperplanes in R^{n_features}"},
    {"Hypothesis class": "Linear SVM", "VC / bound": vc_linear, "Type": "exact VC = d+1",
     "Note": "Same linear separator class"},
    {"Hypothesis class": "Random Forest", "VC / bound": "very large", "Type": "capacity proxy",
     "Note": "Tree ensembles shatter easily; depth controls overfitting"},
    {"Hypothesis class": "CatBoost / boosting", "VC / bound": "very large", "Type": "capacity proxy",
     "Note": "Boosting adds weak learners; regularization via depth, learning rate"},
    {"Hypothesis class": "MiniLM + LogReg", "VC / bound": 385, "Type": "exact VC = d+1",
     "Note": "384-d embeddings + linear head"},
])
print(f"Feature dimension d = {n_features}  |  Training samples n = {n_train:,}")
print(f"Ratio n / VC_linear = {n_train / vc_linear:.0f}  (PAC wants n >> VC)\n")
display(vc_table)

# --- PAC bound for fitted LogReg ---
train_err = 1.0 - accuracy_score(y_train, log_reg.predict(X_train))
test_err = 1.0 - accuracy_score(y_test, log_reg.predict(X_test))
pac_bound = vc_generalization_bound(train_err, vc_linear, n_train)

print(f"\nLogReg — train error: {train_err:.3f}  |  test error: {test_err:.3f}  |  gap: {test_err - train_err:+.3f}")
print(f"PAC/VC upper bound on true error (δ=0.05): {pac_bound:.3f}")

# --- Plot: PAC bound vs sample size ---
ns = np.arange(100, n_train * 2, 100)
bounds = [vc_generalization_bound(0.0, vc_linear, int(n), delta=0.05) for n in ns]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(ns, bounds, color="#6366F1", lw=2)
axes[0].axvline(n_train, color="#EF4444", ls="--", label=f"our n_train = {n_train:,}")
axes[0].set_xlabel("Training sample size n")
axes[0].set_ylabel("PAC upper bound on true error")
axes[0].set_title(f"VC Generalization Bound (VC dim = {vc_linear})", fontweight="bold")
axes[0].set_ylim(0, 1)
axes[0].legend()

# --- Generalization gap across all trained models ---
gap_rows = []
for name, (y_true, y_pred, y_score) in fitted_models.items():
    # Recompute train error where possible (metadata models on X_train)
    if name in ("LogReg (metadata)", "CatBoost (metadata)"):
        if "LogReg" in name:
            tr_pred = log_reg.predict(X_train)
        elif cb is not None:
            tr_pred = cb.predict(X_train).astype(int)
        else:
            tr_err = np.nan
            te_err = 1.0 - accuracy_score(y_true, y_pred)
            gap_rows.append({"model": name.split(" (")[0], "train_error": tr_err, "test_error": te_err, "gap": np.nan})
            continue
        tr_err = 1.0 - accuracy_score(y_train, tr_pred)
    else:
        tr_err = np.nan  # text models: skip train eval for speed
    te_err = 1.0 - accuracy_score(y_true, y_pred)
    gap_rows.append({"model": name.replace(" (metadata)", "").replace(" (text)", "").replace(" (language)", ""),
                     "train_error": tr_err, "test_error": te_err,
                     "gap": te_err - tr_err if not np.isnan(tr_err) else np.nan})

gap_df = pd.DataFrame(gap_rows).dropna(subset=["gap"])
x = np.arange(len(gap_df))
w = 0.35
axes[1].bar(x - w/2, gap_df["train_error"], w, label="Train error", color="#6366F1")
axes[1].bar(x + w/2, gap_df["test_error"], w, label="Test error", color="#EF4444")
axes[1].set_xticks(x)
axes[1].set_xticklabels(gap_df["model"], rotation=25, ha="right", fontsize=8)
axes[1].set_ylabel("Error rate")
axes[1].set_title("Generalization Gap (train vs test)", fontweight="bold")
axes[1].legend()
axes[1].set_ylim(0, max(gap_df["test_error"].max() * 1.2, 0.3))

plt.tight_layout()
plt.savefig(FIG_DIR / "06_learning_theory.png", dpi=160, bbox_inches="tight")
plt.show()

print("\nInterpretation:")
print("- Low VC (LogReg) → tighter PAC bounds, less overfitting risk when n >> VC")
print("- Large gap (test >> train) → model overfits; regularization or simpler class helps")
print("- CatBoost higher capacity → can fit train better but may gap more on test")

## 11) Results — comparison table, model visualizations & explanations

In [ ]:
results_df = pd.DataFrame(results)
results_df["category"] = results_df["model"].map(model_category)
results_df = results_df.sort_values("pr_auc", ascending=False, na_position="last").reset_index(drop=True)

metric_cols = ["accuracy", "balanced_accuracy", "f1", "roc_auc", "pr_auc"]
display_df = results_df[["model", "category"] + metric_cols].round(3)
display(display_df)

results_df.to_csv("final_model_results.csv", index=False)
print("Saved: final_model_results.csv")

In [ ]:
# --- Visualization 1: grouped metric bars by model ---
plot_df = results_df.melt(
    id_vars=["model", "category"],
    value_vars=metric_cols,
    var_name="metric",
    value_name="score",
).dropna(subset=["score"])

metric_labels = {
    "accuracy": "Accuracy",
    "balanced_accuracy": "Balanced Acc",
    "f1": "F1",
    "roc_auc": "ROC-AUC",
    "pr_auc": "PR-AUC",
}
plot_df["metric"] = plot_df["metric"].map(metric_labels)

g = sns.catplot(
    data=plot_df,
    x="score",
    y="model",
    col="metric",
    col_wrap=3,
    kind="bar",
    height=3.2,
    aspect=1.15,
    palette="viridis",
    sharex=False,
)
g.fig.subplots_adjust(top=0.88)
g.fig.suptitle("Model Comparison — All Metrics", fontsize=14, fontweight="bold")
for ax in g.axes.flat:
    ax.set_xlim(0, 1)
    ax.set_xlabel("")
    ax.grid(axis="x", alpha=0.3)
plt.savefig(FIG_DIR / "03_model_metrics_grid.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# --- Visualization 2: ROC + PR curves (models with probabilities) ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = sns.color_palette("tab10", len(fitted_models))

for (name, (y_true, y_pred, y_score)), color in zip(fitted_models.items(), colors):
    if y_score is None or np.all(np.isnan(y_score)):
        continue
    RocCurveDisplay.from_predictions(y_true, y_score, ax=axes[0], name=name, color=color)
    PrecisionRecallDisplay.from_predictions(y_true, y_score, ax=axes[1], name=name, color=color)

axes[0].plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
axes[0].set_title("ROC Curves", fontweight="bold")
axes[1].set_title("Precision-Recall Curves", fontweight="bold")
axes[0].legend(fontsize=8, loc="lower right")
axes[1].legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.savefig(FIG_DIR / "04_roc_pr_curves.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# --- Visualization 3: category-level average performance ---
cat_avg = (
    results_df.groupby("category")[metric_cols]
    .mean(numeric_only=True)
    .reset_index()
    .melt(id_vars="category", var_name="metric", value_name="score")
)
metric_labels = {
    "accuracy": "Accuracy",
    "balanced_accuracy": "Balanced Acc",
    "f1": "F1",
    "roc_auc": "ROC-AUC",
    "pr_auc": "PR-AUC",
}
cat_avg["metric"] = cat_avg["metric"].map(metric_labels)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=cat_avg, x="metric", y="score", hue="category", palette="Set2", ax=ax)
ax.set_ylim(0, 1)
ax.set_title("Average Performance by Model Category", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Score")
ax.legend(title="Category", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(FIG_DIR / "05_category_averages.png", dpi=160, bbox_inches="tight")
plt.show()

### Results explanation

Run the cell below after all models finish — it auto-generates interpretations for your report.

In [ ]:
def explain_results(df: pd.DataFrame) -> str:
    best = df.sort_values("pr_auc", ascending=False).iloc[0]
    best_acc = df.sort_values("accuracy", ascending=False).iloc[0]
    meta = df[df["category"] == "Metadata"]
    lang = df[df["category"] == "Language"]

    lines = [
        "# Summary Report — Movie Rating Prediction\n",
        f"**Best overall (PR-AUC):** {best['model']} — PR-AUC={best['pr_auc']:.3f}, ROC-AUC={best['roc_auc']:.3f}, Acc={best['accuracy']:.1%}, F1={best['f1']:.3f}",
        f"**Highest accuracy:** {best_acc['model']} — {best_acc['accuracy']:.1%}\n",
        "## Baselines",
        "- **Uniform random (~50% acc):** Minimum bar; any useful model must beat this on balanced accuracy and F1.",
        "- **Majority class (~67% acc, 0% F1):** High accuracy is misleading — never predicts high-rated movies.\n",
        "## Metadata models",
    ]

    if len(meta):
        m_best = meta.sort_values("pr_auc", ascending=False).iloc[0]
        lines.append(
            f"- **{m_best['model']}** leads with **{m_best['accuracy']:.1%} accuracy**, PR-AUC **{m_best['pr_auc']:.3f}**."
        )
        lines.append("- Popularity and vote count are strong proxies for audience reception.")

    lines.append("\n## Language models")

    if len(lang):
        l_best = lang.sort_values("pr_auc", ascending=False).iloc[0]
        lines.append(
            f"- **Best language model:** {l_best['model']} — Acc={l_best['accuracy']:.1%}, PR-AUC={l_best['pr_auc']:.3f}."
        )
        lines.append("- **TF-IDF** captures keywords but misses deeper semantics.")
        lines.append("- **LSTM from scratch** underperforms — must learn vocabulary from ~10k examples.")
        lines.append("- **MiniLM** uses pretrained semantic embeddings for true language understanding.")

    if len(meta) and len(lang):
        meta_best_acc = meta["accuracy"].max()
        lang_best_acc = lang["accuracy"].max()
        if meta_best_acc > lang_best_acc:
            lines.append(
                f"\n> **Key finding:** Metadata ({meta_best_acc:.1%}) beats language-only ({lang_best_acc:.1%}). "
                "Overviews describe plot, not rating — structured signals dominate."
            )
        else:
            lines.append(
                f"\n> **Key finding:** Language ({lang_best_acc:.1%}) beats metadata ({meta_best_acc:.1%})."
            )

    lines.append("\n## Recommendation")
    lines.append(f"- Primary model: **{best['model']}**")
    lines.append("- Prioritize PR-AUC and F1 over raw accuracy (67% majority class).")

    return "\n".join(lines)

summary_md = explain_results(results_df)
print(summary_md)

Path("SUMMARY_REPORT.md").write_text(summary_md, encoding="utf-8")
print("\nSaved: SUMMARY_REPORT.md")